Documentation:

[Scipy documentation on related topic](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.wilcoxon.html)

Youtube materials on related topic

[Wilcoxon-Test(Wilcoxon Signed Rank Test)
](https://www.youtube.com/watch?v=NZsL2eDQiDQ)

[Wilcoxon Test (Wilcoxon signed-rank test) — Simply explained
](https://www.youtube.com/watch?v=2AqoK8itEFQ)

# Intro: 

In [2]:
from scipy import stats

# Input: Two lists of the same length
before = [8, 7, 9, 6, 8]
after  = [4, 5, 3, 4, 2]

# The function handles the subtraction and ranking for you
statistic, p_value = stats.wilcoxon(before, after)

print(f"Test Statistic: {statistic}")
print(f"P-value: {p_value}")


Test Statistic: 0.0
P-value: 0.0625


# Scenario 1: Did students’ exam scores improve after a 4-week study program?

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon, shapiro

# Synthetic dataset for study purpose

df = pd.DataFrame({
    "student_id": range(1, 21),

    # Exam scores before the study program
    "score_before": [
        58, 62, 65, 70, 72,
        74, 75, 78, 80, 82,
        83, 85, 88, 90, 92,
        68, 76, 79, 81, 84
    ],

    "score_after": [
        63, 66, 67, 74, 72,
        78, 79, 81, 84, 85,
        83, 90, 91, 94, 97,
        75, 76, 83, 80, 96
    ]
})

# Create paired difference column
df["difference"] = df["score_after"] - df["score_before"]

print(df)

    student_id  score_before  score_after  difference
0            1            58           63           5
1            2            62           66           4
2            3            65           67           2
3            4            70           74           4
4            5            72           72           0
5            6            74           78           4
6            7            75           79           4
7            8            78           81           3
8            9            80           84           4
9           10            82           85           3
10          11            83           83           0
11          12            85           90           5
12          13            88           91           3
13          14            90           94           4
14          15            92           97           5
15          16            68           75           7
16          17            76           76           0
17          18            79

In [3]:
# Basic descriptive statistics
print(df[["score_before", "score_after", "difference"]].describe())

# Count improvement / no change / decrease
positive_changes = (df["difference"] > 0).sum()
zero_changes = (df["difference"] == 0).sum()
negative_changes = (df["difference"] < 0).sum()

print("Improved students:", positive_changes)
print("No change:", zero_changes)
print("Decreased students:", negative_changes)

       score_before  score_after  difference
count     20.000000    20.000000   20.000000
mean      77.100000    80.700000    3.600000
std        9.227533     9.766215    2.835861
min       58.000000    63.000000   -1.000000
25%       71.500000    74.750000    2.750000
50%       78.500000    80.500000    4.000000
75%       83.250000    86.250000    4.250000
max       92.000000    97.000000   12.000000
Improved students: 16
No change: 3
Decreased students: 1


In [4]:
stat, p = shapiro(df["difference"])

print("Shapiro-Wilk statistic:", stat)
print("p-value:", p)

if p < 0.05:
    print("The paired differences are not normally distributed.")
else:
    print("The paired differences do not strongly violate normality.")

Shapiro-Wilk statistic: 0.8712512573264061
p-value: 0.01236012133154475
The paired differences are not normally distributed.


Since our research question is directional - Did scores improve after the program? we use alternative="greater"

In [ ]:
stat, p = wilcoxon(
    df["score_after"],
    df["score_before"],
    alternative="greater",
    zero_method="wilcox",
    method="auto"
)

print("Wilcoxon statistic:", stat)
print("p-value:", p)

Wilcoxon statistic: 203.0
p-value: 0.00011640347810628148


# Note: 
# If a student gets the exact same score "Before" and "After," the difference is zero. You have to decide what to do with them:

# zero_method='wilcox' (Default): It completely deletes those students from the test.

# zero_method='pratt': It keeps them in the ranking process but then drops them later (a bit more cautious).

# zero_method='zsplit': It splits the "neutral" rank between the positive and negative sides.


In [6]:
alpha = 0.05

if p < alpha:
    print("Reject H0: Exam scores significantly improved after the study program.")
else:
    print("Fail to reject H0: There is not enough evidence that exam scores improved.")

Reject H0: Exam scores significantly improved after the study program.


# The Wilcoxon signed-rank test showed a statistically significant improvement in students’ exam scores after the 4-week study program, W = 152.0, p < .001. Therefore, we reject the null hypothesis and conclude that students’ post-program scores were significantly higher than their pre-program scores.

# Scenario 2: Did employees’ stress levels decrease after a workplace meditation session?

In [8]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon, shapiro

# ---------------------------------------------------------
# 1. Create synthetic paired ordinal dataset
# ---------------------------------------------------------

df = pd.DataFrame({
    "employee_id": range(1, 26),

    # Stress level before meditation session
    # Scale: 1 = very low stress, 10 = very high stress
    "stress_before": [
        8, 7, 9, 6, 8,
        7, 10, 6, 5, 9,
        8, 7, 6, 9, 10,
        5, 7, 8, 6, 9,
        7, 8, 6, 5, 10
    ],

    # Stress level after meditation session
    # Includes:
    # - many repeated values
    # - several decreases
    # - some zero differences
    # - one employee got worse
    "stress_after": [
        6, 5, 7, 5, 6,
        6, 8, 4, 5, 7,
        6, 7, 5, 6, 8,
        4, 5, 6, 6, 8,
        5, 7, 4, 6, 7
    ]
})

# ---------------------------------------------------------
# 2. Create paired difference column
# ---------------------------------------------------------

# Negative difference means stress decreased
df["difference"] = df["stress_after"] - df["stress_before"]

print(df)

    employee_id  stress_before  stress_after  difference
0             1              8             6          -2
1             2              7             5          -2
2             3              9             7          -2
3             4              6             5          -1
4             5              8             6          -2
5             6              7             6          -1
6             7             10             8          -2
7             8              6             4          -2
8             9              5             5           0
9            10              9             7          -2
10           11              8             6          -2
11           12              7             7           0
12           13              6             5          -1
13           14              9             6          -3
14           15             10             8          -2
15           16              5             4          -1
16           17              7 

In [9]:
print(df[["stress_before", "stress_after", "difference"]].describe())

decreased = (df["difference"] < 0).sum()
no_change = (df["difference"] == 0).sum()
increased = (df["difference"] > 0).sum()

print("Employees with decreased stress:", decreased)
print("Employees with no change:", no_change)
print("Employees with increased stress:", increased)

       stress_before  stress_after  difference
count      25.000000     25.000000   25.000000
mean        7.440000      5.960000   -1.480000
std         1.583246      1.206924    0.962635
min         5.000000      4.000000   -3.000000
25%         6.000000      5.000000   -2.000000
50%         7.000000      6.000000   -2.000000
75%         9.000000      7.000000   -1.000000
max        10.000000      8.000000    1.000000
Employees with decreased stress: 21
Employees with no change: 3
Employees with increased stress: 1


In [10]:
print("Stress before value counts:")
print(df["stress_before"].value_counts().sort_index())

print("\nStress after value counts:")
print(df["stress_after"].value_counts().sort_index())

print("\nDifference value counts:")
print(df["difference"].value_counts().sort_index())

Stress before value counts:
stress_before
5     3
6     5
7     5
8     5
9     4
10    3
Name: count, dtype: int64

Stress after value counts:
stress_after
4    3
5    6
6    8
7    5
8    3
Name: count, dtype: int64

Difference value counts:
difference
-3     2
-2    13
-1     6
 0     3
 1     1
Name: count, dtype: int64


In [11]:
Q1 = df["difference"].quantile(0.25)
Q3 = df["difference"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df["difference"] < lower_bound) |
    (df["difference"] > upper_bound)
]

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print(outliers)

Lower bound: -3.5
Upper bound: 0.5
    employee_id  stress_before  stress_after  difference
23           24              5             6           1


In [12]:
stat, p = shapiro(df["difference"])

print("Shapiro-Wilk statistic:", stat)
print("p-value:", p)

if p < 0.05:
    print("The paired differences are not normally distributed.")
else:
    print("The paired differences do not strongly violate normality.")

Shapiro-Wilk statistic: 0.851810588083601
p-value: 0.0019066149939621334
The paired differences are not normally distributed.


In [ ]:
stat, p = wilcoxon(
    df["stress_after"],
    df["stress_before"],
    alternative="less",# since we investigate decrease case
    zero_method="wilcox", # remove pairs where the difference is exactly zero before ranking.
    method="auto"  # if ties/repeated ranks exist, SciPy will adjust the method automatically.
)

print("Wilcoxon statistic:", stat)
print("p-value:", p)

Wilcoxon statistic: 4.0
p-value: 2.1369386728799973e-05


In [14]:
alpha = 0.05

if p < alpha:
    print("Reject H0: Stress levels significantly decreased after meditation.")
else:
    print("Fail to reject H0: There is not enough evidence that stress decreased.")

Reject H0: Stress levels significantly decreased after meditation.


# Note: 
alternative="less"
→ first input is smaller than second input
→ after < before
→ decrease

alternative="greater"
→ first input is larger than second input
→ after > before
→ increase

alternative="two-sided"
→ first input is different from second input
→ either decrease or increase

# Scenario 4: Did patient symptom scores change after treatment?
Note: Now the main question is not directional - decrease/increase but rather change. 
alternative="two-sided" will be used.

In [1]:
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------
# 1. Create synthetic paired clinical-style dataset
# ---------------------------------------------------------

df = pd.DataFrame({
    "patient_id": range(1, 37),

    "severity_group": [
        "Mild", "Mild", "Mild", "Mild", "Mild", "Mild",
        "Mild", "Mild", "Mild", "Mild", "Mild", "Mild",

        "Moderate", "Moderate", "Moderate", "Moderate", "Moderate", "Moderate",
        "Moderate", "Moderate", "Moderate", "Moderate", "Moderate", "Moderate",

        "Severe", "Severe", "Severe", "Severe", "Severe", "Severe",
        "Severe", "Severe", "Severe", "Severe", "Severe", "Severe"
    ],

    # Higher score = worse symptoms
    "symptom_before": [
        8, 9, 10, 11, 12, 10, 9, 8, 11, 12, 10, 9,
        15, 16, 18, 17, 19, 20, 16, 18, 21, 19, 17, 20,
        24, 26, 28, 27, 30, 29, 25, 31, 28, 32, 27, 30
    ],

    # Includes repeated values, zero changes, and a few worsening cases
    "symptom_after": [
        7, 9, 8, 10, 12, 9, 9, 7, 10, 11, 10, 8,
        12, 15, 14, 18, 16, 17, 16, 15, 18, 20, 14, 18,
        19, 22, 21, 25, 24, 23, 25, 26, 20, 30, 24, 25
    ]
})

# ---------------------------------------------------------
# 2. Calculate paired change
# ---------------------------------------------------------

df["change"] = df["symptom_after"] - df["symptom_before"]

print("Dataset:")
print(df)

print("\nChange meaning:")
print("change < 0  → symptoms decreased")
print("change = 0  → no change")
print("change > 0  → symptoms increased/worsened")

Dataset:
    patient_id severity_group  symptom_before  symptom_after  change
0            1           Mild               8              7      -1
1            2           Mild               9              9       0
2            3           Mild              10              8      -2
3            4           Mild              11             10      -1
4            5           Mild              12             12       0
5            6           Mild              10              9      -1
6            7           Mild               9              9       0
7            8           Mild               8              7      -1
8            9           Mild              11             10      -1
9           10           Mild              12             11      -1
10          11           Mild              10             10       0
11          12           Mild               9              8      -1
12          13       Moderate              15             12      -3
13          14       Mode

In [2]:
print("\nOverall descriptive statistics:")
print(df[["symptom_before", "symptom_after", "change"]].describe())

print("\nOverall change summary:")
print("Decreased symptoms:", (df["change"] < 0).sum())
print("No change:", (df["change"] == 0).sum())
print("Increased symptoms:", (df["change"] > 0).sum())

print("\nChange value counts:")
print(df["change"].value_counts().sort_index())


Overall descriptive statistics:
       symptom_before  symptom_after     change
count       36.000000      36.000000  36.000000
mean        18.666667      16.305556  -2.361111
std          7.768066       6.413465   2.269711
min          8.000000       7.000000  -8.000000
25%         11.000000      10.000000  -3.250000
50%         18.000000      16.000000  -2.000000
75%         26.250000      21.250000  -1.000000
max         32.000000      30.000000   1.000000

Overall change summary:
Decreased symptoms: 28
No change: 6
Increased symptoms: 2

Change value counts:
change
-8    1
-7    1
-6    2
-5    3
-4    2
-3    7
-2    4
-1    8
 0    6
 1    2
Name: count, dtype: int64


In [3]:
group_summary = df.groupby("severity_group").agg(
    n=("patient_id", "count"),
    median_before=("symptom_before", "median"),
    median_after=("symptom_after", "median"),
    median_change=("change", "median"),
    decreased=("change", lambda x: (x < 0).sum()),
    no_change=("change", lambda x: (x == 0).sum()),
    increased=("change", lambda x: (x > 0).sum())
)

print("\nGroup-level summary:")
print(group_summary)


Group-level summary:
                 n  median_before  median_after  median_change  decreased  \
severity_group                                                              
Mild            12           10.0           9.0           -1.0          8   
Moderate        12           18.0          16.0           -3.0          9   
Severe          12           28.0          24.0           -5.0         11   

                no_change  increased  
severity_group                        
Mild                    4          0  
Moderate                1          2  
Severe                  1          0  


In [4]:
overall_w, overall_p = wilcoxon(
    df["symptom_after"],
    df["symptom_before"],
    alternative="two-sided",
    zero_method="wilcox",
    method="auto"
)

print("\nOverall Wilcoxon signed-rank test:")
print("W statistic:", overall_w)
print("p-value:", overall_p)

if overall_p < 0.05:
    print("Conclusion: Symptom scores changed significantly after treatment overall.")
else:
    print("Conclusion: There is not enough evidence of a significant overall change.")


Overall Wilcoxon signed-rank test:
W statistic: 11.0
p-value: 4.546214415747873e-06
Conclusion: Symptom scores changed significantly after treatment overall.
